In [0]:
# ==============================================================
# Notebook : 01_pos_silver.py
# Purpose  : Bronze → Silver for POS streaming transactions
# Source   : bronze/stream/pos_transactions/
# Target   : silver/fact_pos_transactions/ (Delta)
# =============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    DoubleType, IntegerType, BooleanType, TimestampType, ArrayType
)
from delta.tables import DeltaTable
from datetime import datetime

# ── Config ────────────────────────────────────────────────────
BRONZE_PATH = "abfss://bronze@walmartdata.dfs.core.windows.net/stream/pos_transactions/*/*/"
SILVER_PATH = "abfss://silver@walmartdata.dfs.core.windows.net/fact_pos_transactions/"
CHECKPOINT  = "abfss://silver@walmartdata.dfs.core.windows.net/_checkpoints/pos_transactions/"

print("=" * 60)
print("  POS Transactions: Bronze → Silver")
print(f"  Started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)

In [0]:
# ── CELL 2: Read raw Bronze POS JSON ─────────────────────────
df_raw = spark.read.option("multiLine", True).json(BRONZE_PATH)

print(f"Raw Bronze rows    : {df_raw.count():,}")
print(f"Raw Bronze columns : {len(df_raw.columns)}")
print("\nRaw schema:")
df_raw.printSchema()

In [0]:
# ── CELL 3: Flatten + enforce schema ─────────────────────────
# POS events have nested 'items' array — we keep transaction
# level here and explode items in Gold layer

df_typed = (
    df_raw
    .select(
        F.col("transaction_id").cast(StringType()),
        F.col("store_id").cast(StringType()),
        F.col("store_city").cast(StringType()),
        F.col("store_region").cast(StringType()),
        F.col("terminal_id").cast(StringType()),
        F.to_timestamp("timestamp").alias("transaction_ts"),
        F.col("cashier_id").cast(StringType()),
        F.col("customer_loyalty_id").cast(StringType()),
        F.col("payment_method").cast(StringType()),
        F.col("item_count").cast(IntegerType()),
        F.col("subtotal").cast(DoubleType()),
        F.col("tax_amount").cast(DoubleType()),
        F.col("total_amount").cast(DoubleType()),
        F.col("transaction_type").cast(StringType()),
        #F.col("items"),   # keep array for now
        F.to_timestamp("ingested_at").alias("ingested_at"),
        # ── Audit columns ──────────────────────────────────
        F.current_timestamp().alias("silver_processed_at"),
        F.lit("pos_bronze_to_silver_v1").alias("pipeline_version"),
    )
)

print(f"Typed rows: {df_typed.count():,}")
df_typed.printSchema()

In [0]:
# ── CELL 4: Data quality checks ──────────────────────────────
total_rows = df_typed.count()

checks = {
    "null_transaction_id" : df_typed.filter(F.col("transaction_id").isNull()).count(),
    "null_store_id"       : df_typed.filter(F.col("store_id").isNull()).count(),
    "null_timestamp"      : df_typed.filter(F.col("transaction_ts").isNull()).count(),
    "negative_total"      : df_typed.filter(F.col("total_amount") <= 0).count(),
    "null_payment"        : df_typed.filter(F.col("payment_method").isNull()).count(),
    "future_transactions" : df_typed.filter(F.col("transaction_ts") > F.current_timestamp()).count(),
}

print("=" * 50)
print("  DATA QUALITY REPORT — POS Transactions")
print("=" * 50)
print(f"  Total rows        : {total_rows:,}")
for check, count in checks.items():
    status = "✅" if count == 0 else "⚠️ "
    pct    = round(count / total_rows * 100, 2) if total_rows > 0 else 0
    print(f"  {status} {check:<25}: {count:,}  ({pct}%)")
print("=" * 50)


In [0]:
# ── CELL 5: Deduplicate ───────────────────────────────────────
# Keep latest record per transaction_id in case of duplicates
# from Event Hubs at-least-once delivery guarantee
from pyspark.sql.window import Window
df_deduped = (
    df_typed
    .withColumn(
        "row_num",
        F.row_number().over(
            Window.partitionBy("transaction_id")
                  .orderBy(F.col("ingested_at").desc())
        )
    )
    .filter(F.col("row_num") == 1)
    .drop("row_num")
)

duplicates_removed = df_typed.count() - df_deduped.count()
print(f"Rows before dedup : {df_typed.count():,}")
print(f"Rows after dedup  : {df_deduped.count():,}")
print(f"Duplicates removed: {duplicates_removed:,}")

In [0]:
# ── CELL 6: Filter out bad records ───────────────────────────
df_clean = (
    df_deduped
    .filter(F.col("transaction_id").isNotNull())
    .filter(F.col("store_id").isNotNull())
    .filter(F.col("total_amount") > 0)
    .filter(F.col("transaction_ts").isNotNull())
    # Add derived columns
    .withColumn("sale_date",  F.to_date("transaction_ts"))
    .withColumn("sale_hour",  F.hour("transaction_ts"))
    .withColumn("sale_month", F.month("transaction_ts"))
    .withColumn("sale_year",  F.year("transaction_ts"))
    .withColumn("is_loyalty_transaction",
                F.col("customer_loyalty_id").isNotNull())
    .withColumn("basket_size_category",
                F.when(F.col("total_amount") < 200,  "Small")
                 .when(F.col("total_amount") < 1000, "Medium")
                 .when(F.col("total_amount") < 5000, "Large")
                 .otherwise("Premium"))
)

print(f"Clean rows: {df_clean.count():,}")
display(df_clean.limit(5))

In [0]:
# ── CELL 7: Write to Silver as Delta ─────────────────────────
from pyspark.sql.window import Window

(
    df_clean
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("sale_year", "sale_month", "store_region")
    .save(SILVER_PATH)
)

print(f"✅ Written to Silver: {SILVER_PATH}")
print(f"   Partitioned by  : sale_year / sale_month / store_region")

In [0]:
# ── CELL 8: Optimize Delta table ─────────────────────────────
# Z-ORDER on most common query columns for faster reads
spark.sql(f"""
    OPTIMIZE delta.`{SILVER_PATH}`
    ZORDER BY (store_id, sale_date, payment_method)
""")

print("✅ Delta table optimized with Z-ORDER")

In [0]:
# ── CELL 9: Register in Unity Catalog ────────────────────────
spark.sql("CREATE DATABASE IF NOT EXISTS walmart_silver")

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS walmart_silver.fact_pos_transactions
    USING DELTA
    LOCATION '{SILVER_PATH}'
""")

# Final row count from registered table
final_count = spark.sql("""
    SELECT COUNT(*) AS total_rows
    FROM walmart_silver.fact_pos_transactions
""").collect()[0]["total_rows"]

print(f"✅ Registered in Unity Catalog: walmart_silver.fact_pos_transactions")
print(f"   Final row count: {final_count:,}")

# Quick summary stats
spark.sql("""
    SELECT
        store_region,
        COUNT(*)            AS transactions,
        SUM(total_amount)   AS total_revenue,
        AVG(total_amount)   AS avg_basket,
        SUM(CASE WHEN is_loyalty_transaction
                 THEN 1 ELSE 0 END) AS loyalty_txns
    FROM walmart_silver.fact_pos_transactions
    GROUP BY store_region
    ORDER BY total_revenue DESC
""").display()